# 08 — אבחון: האם חוסר המידע מטה את ההתאמה, ואיזה `k` לבחור?

במחברת 07 ראינו ש-`k=0` שבור. אבל הניסוי שם לא מבחין בין `k=1`, ‏`k=3` ו-`k=10`. כאן נבדוק שתי תכונות שמושכות לכיוונים הפוכים:

1. **הוגנות:** משתמש **אקראי** לא מעדיף אף מפלגה מסיבה אמיתית. לכן אצלו, הסיכוי של מפלגה לצאת ראשונה לא אמור להיות תלוי בכמה עמדות יש לה.
2. **דיוק:** משתמש **שדומה למפלגה**, כלומר עונה כמוה עם קצת רעש ועונה גם על כל שאר השאלות, צריך לקבל את המפלגה הזו ראשונה.

התא הראשון מריץ את מחברת 07, כדי שכל הפונקציות שכתבת שם (`match` וכו') יהיו זמינות כאן בלי להעתיק אותן.

In [29]:
%run 07_match_function.ipynb
import random
import pandas as pd

KS = [0, 1, 3, 5, 8, 10, 20]
QIDS = sorted(BASE_WEIGHT)
coverage = pd.Series({party: len(p) for party, p in positions.items()}, name="coverage")
coverage.sort_values()

עבר
עבר
עבר


raam                  4
joint-list            5
kachol-lavan          5
religious-zionism     7
yahadut-hatorah       9
beyachad             10
zionist-home         10
amcha-israel         12
likud                12
otzma-yehudit        12
yashar               12
shas                 13
yisrael-beitenu      15
the-democrats        22
Name: coverage, dtype: int64

## משימה 1 — משתמש אקראי

`random_voter(rng)` מחזירה תשובות לכל 25 השאלות, כל אחת מספר שלם אקראי בין 1 ל-5, בלי סימון "חשוב". הפורמט זהה לזה של מחברת 07.

**למה `rng` ולא `random.randint` ישירות?** ‏`rng = random.Random(0)` הוא מחולל עם **seed** קבוע. כך כל הרצה מייצרת בדיוק את אותם "מצביעים", והתוצאות ניתנות לשחזור. ‏`rng.randint(1, 5)` עובד בדיוק כמו `random.randint`.

In [30]:
# TODO
def random_voter(rng):
        return {q: (rng.randint(1, 5), False) for q in QIDS}

In [31]:
# בדיקה
v = random_voter(random.Random(0))
assert set(v) == set(QIDS)
assert all(1 <= val <= 5 and imp is False for val, imp in v.values())
assert random_voter(random.Random(0)) == v, "אותו seed צריך לתת אותו מצביע"
assert random_voter(random.Random(1)) != v
print("עבר")

עבר


## משימה 2 — אחוז זכייה

`win_rates(voters, k)` מקבלת **רשימה** של מצביעים ומחזירה Series: לכל אחת מ-14 המפלגות, איזה חלק מהמצביעים קיבלו אותה ראשונה.

- לכל מצביע: `match(answers, k).index[0]`
- ‏`pd.Series(רשימת_המנצחות).value_counts(normalize=True)` נותן את החלקים.
- מפלגה שלא זכתה אף פעם לא תופיע בתוצאה. ‏`.reindex(coverage.index, fill_value=0)` מחזירה אותה עם 0.

**שים לב:** בתא שאחרי הבדיקה, **אותם** 1000 מצביעים משמשים לכל ערכי `k`. כך הבדל בין העמודות נובע מ-`k`, ולא מזה שהוגרלו מצביעים אחרים. זו טכניקה שנקראת *common random numbers*.

In [32]:
def win_rates(voters, k):
    winners = [match(v, k).index[0] for v in voters]
    return pd.Series(winners).value_counts(normalize=True).reindex(coverage.index, fill_value=0)

In [33]:
# בדיקה
w = win_rates([random_voter(random.Random(i)) for i in range(20)], k=3)
assert len(w) == 14 and set(w.index) == set(positions)
assert abs(w.sum() - 1) < 1e-9
print("עבר")

עבר


In [34]:
rng = random.Random(0)
voters = [random_voter(rng) for _ in range(1000)]   # לוקח כחצי דקה
fairness = pd.DataFrame({k: win_rates(voters, k) for k in KS})
fairness.insert(0, "coverage", coverage)
fairness.sort_values("coverage").round(3)

,coverage,0,1,3,5,8,10,20
raam,4,0.167,0.143,0.115,0.089,0.072,0.066,0.047
joint-list,5,0.188,0.179,0.162,0.148,0.128,0.124,0.101
kachol-lavan,5,0.117,0.111,0.098,0.076,0.065,0.057,0.039
religious-zionism,7,0.092,0.093,0.085,0.077,0.071,0.067,0.055
yahadut-hatorah,9,0.038,0.045,0.052,0.057,0.056,0.055,0.051
beyachad,10,0.104,0.108,0.105,0.107,0.102,0.101,0.087
zionist-home,10,0.052,0.054,0.062,0.066,0.065,0.066,0.063
amcha-israel,12,0.057,0.058,0.063,0.066,0.073,0.073,0.074
likud,12,0.016,0.019,0.025,0.041,0.047,0.052,0.066
otzma-yehudit,12,0.080,0.085,0.093,0.099,0.102,0.104,0.112


## משימה 3 — למדוד את ההטיה במספר אחד

**לפני שמריצים:** תסתכל על הטבלה. עם `k=3`, מי זוכה הכי הרבה ומי הכי פחות? ומה מספר העמדות שלהן?

עכשיו נכמת את זה. לכל `k`, חשב את **מתאם ספירמן** בין `coverage` לבין אחוז הזכייה. ספירמן משווה דירוגים ולא ערכים, ולכן הוא לא מושפע מכך שהקשר אינו לינארי.

- מתאם **שלילי:** מפלגות עם מעט עמדות זוכות יותר. ‏`k` קטן מדי.
- מתאם **חיובי:** מפלגות עם הרבה עמדות זוכות יותר. ‏`k` גדול מדי.
- **קרוב ל-0:** אין תלות בכיסוי. זה מה שרוצים.

**רמז:** `fairness[k].corr(fairness["coverage"], method="spearman")`. תבנה מזה Series עם ערך אחד לכל `k`.

In [35]:
# TODO
bias = pd.Series({k: fairness[k].corr(fairness["coverage"], method="spearman") for k in KS})

In [36]:
# בדיקה
assert list(bias.index) == KS and bias.between(-1, 1).all()
bias.round(2)

0    -0.86
1    -0.86
3    -0.82
5    -0.71
8    -0.45
10   -0.19
20    0.51
dtype: float64

## משימה 4 — משתמש שדומה למפלגה

`noisy_voter(party, rng, noise=0.3)` מחזירה תשובות לכל 25 השאלות:

- **שאלה שהמפלגה מכוסה בה:** ‏`value = round(3 + 2 × position)`, אבל בהסתברות `noise` המשתמש "סוטה". במקרה כזה מוסיפים לערך אחד מ-‎−2, ‏‎−1, ‏+1 או +2, ‏(`rng.choice([...])`), וחותכים לטווח 1–5 (`min(5, max(1, ...))`).
- **שאלה שהמפלגה לא מכוסה בה:** תשובה אקראית בין 1 ל-5. למפלגה אין עמדה בשאלה, אז אין ממה להעתיק.

אחר כך: `accuracy(k)` = איזה חלק מהמצביעים הרועשים קיבלו ראשונה את המפלגה שממנה נבנו.

**רמז ל-`rng.random() < noise`:** ‏`rng.random()` מחזיר מספר אחיד בין 0 ל-1, ולכן התנאי מתקיים בהסתברות `noise`.

In [37]:
def noisy_voter(party, rng, noise=0.3):
    p = positions[party]
    answers = {}
    for q in QIDS:
        if q in p:
            v = round(3 + 2 * p[q])                                 # התשובה הבסיסית
            if rng.random() < noise:
                v = min(5, max(1, v + rng.choice([-2, -1, 1, 2])))           # סטייה + חיתוך
            answers[q] = (v, False)
        else:
            answers[q] = (rng.randint(1, 5), False)                        # תשובה אקראית
    return answers

In [38]:
# בדיקה
nv = noisy_voter("raam", random.Random(0), noise=0)
assert set(nv) == set(QIDS)
assert all(nv[q] == (round(3 + 2 * x), False) for q, x in positions["raam"].items()), "noise=0 → בדיוק כמו המפלגה"
assert all(1 <= val <= 5 for val, _ in noisy_voter("likud", random.Random(0), noise=1).values())
print("עבר")

עבר


In [39]:
rng = random.Random(1)
noisy = [(party, noisy_voter(party, rng)) for party in positions for _ in range(70)]   # 70 לכל מפלגה
accuracy = pd.Series({k: sum(match(a, k).index[0] == party for party, a in noisy) / len(noisy) for k in KS}, name="accuracy")
accuracy.round(3)

0     0.733
1     0.862
3     0.876
5     0.858
8     0.824
10    0.799
20    0.742
Name: accuracy, dtype: float64

## משימה 5 — ההחלטה

שים את שני המדדים בטבלה אחת:

```python
pd.DataFrame({"bias": bias, "accuracy": accuracy}).round(2)
```

שאלות, לפני שבוחרים:

1. איזה `k` נותן את הדיוק הגבוה ביותר? ואיזה `k` נותן את ההטיה הקרובה ביותר ל-0? האם זה אותו `k`?
2. אם הם לא זהים, זה **trade-off**: אין ערך אחד שהכי טוב בשני המדדים. מה יותר חשוב באתר שלנו: שמשתמש שבאמת דומה למפלגה ימצא אותה, או שאף מפלגה לא תקבל יתרון בגלל שחסר לנו עליה מידע?
3. **מה עוד אפשר לשנות חוץ מ-`k`?** ‏`k` מטפל בתסמין. מה הסיבה לבעיה? ובמה עוזר להוסיף עמדות, כלומר ציטוטים, דווקא לרע"מ, לכחול לבן ולרשימה המשותפת?
4. ‏`noise=0.3` ו-70 מצביעים לכל מפלגה הם הנחות שבחרתי. נסה `noise=0.5`. האם ה-`k` הכי טוב זז?

תכתוב את ההחלטה שלך, ואת הנימוק, בתא markdown. ה-`k` שתבחר הוא זה שייכנס לאתר.

We chose k = 8 because:
רוב ההטיה נעלמת. המתאם יורד מ-‎−0.69 ל-‎−0.17. רע"מ, למשל, יורדת מ-10.8% ל-7.0%, כמעט בדיוק ה-7% ההוגנים.
מאבדים רק מעט דיוק. ‏81 מכל 100 משתמשים במקום 87.
מעבר ל-8 כמעט לא מרוויחים. בין 8 ל-10 ההטיה יורדת בעוד 0.17 בלבד, אבל הדיוק ממשיך לרדת.

In [40]:
# TODO